In [5]:
from models.LLMs import GPT_4o
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from utils.tools import Searxng
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal, Annotated, TypedDict, List
from langgraph.graph import StateGraph, add_messages
from utils.custom_output_parser import CustomOutputParser
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor, create_handoff_tool
import uuid, json, os
import requests, datetime, logging
from langchain_core.tools import tool, InjectedToolCallId
from langchain_core.runnables import RunnableConfig
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, END, START
from langchain_core.messages import BaseMessage, ToolMessage, HumanMessage
from langchain_core.messages import ToolMessage
from utils.tools import image_search, web_search, crawl_url, generate_presentation_outline, generate_presentation
import operator
import pprint

OUTPUT_DIR = os.path.join(os.getcwd(), "semi_output")
GENERATED_SLIDES_DIR = os.path.join(os.getcwd(), "generated_slides")
LLM = GPT_4o()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AgentState(TypedDict):
    messages: List[BaseMessage]
    remaining_steps: int
    is_last_step: bool
    outline: str
    images: list[dict]
    found_information: list[dict]
    input: str
    next_step: str

members = ["outline_agent", "slide_agent"]
options = members + ["FINISH"]

class Router(TypedDict):
    next: Literal["outline_agent", "slide_agent", "FINISH"]

def supervisor_node(state: AgentState) -> Command[Literal["outline_agent", "slide_agent", "__end__"]]:
    system_prompt = f"""
    You are a supervisor, tasked with managing a conversation between the following workers: {members}. 
    Given the user's request, respond with the worker to act next. 
    Each worker will perform a task and respond with results and status. 
    When finished, respond with FINISH.
    """
    messages = [{"role": "system", "content": system_prompt}] + state["messages"]
    response = LLM.with_structured_output(Router).invoke(messages)
    goto = response["next"]
    if goto == "FINISH":
        goto = END
    return Command(goto=goto, update={"next": goto})

def outline_agent_node(state: AgentState) -> Command[Literal["supervisor"]]:
    outline_agent = create_react_agent(
        model=LLM,
        tools=[image_search, crawl_url, web_search, generate_presentation_outline],
    )
    result = outline_agent.invoke(state)
    return Command(
        update={"messages": [
            HumanMessage(content=result["messages"][-1].content, name="outline_agent")
        ]},
        goto="supervisor"
    )
    
def slide_agent_node(state: AgentState) -> Command[Literal["supervisor"]]:
    slide_agent = create_react_agent(
        model=LLM,
        tools=[generate_presentation],
    )
    result = slide_agent.invoke(state)
    return Command(
        update={"messages": [
            HumanMessage(content=result["messages"][-1].content, name="slide_agent")
        ]},
        goto="supervisor"
    )

graph = StateGraph(AgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("outline_agent", outline_agent_node)
graph.add_node("slide_agent", slide_agent_node)

graph.add_edge(START, "supervisor")

app = graph.compile()
config = {"configurable": {"thread_id": "1"}}


In [12]:
from IPython.display import display, Markdown

# Get the Mermaid syntax for the graph
mermaid_syntax = app.get_graph().draw_mermaid()

# Display it using Markdown which will render it if the notebook has mermaid support
mermaid_markdown = f"""
```mermaid
{mermaid_syntax}
```
"""

display(Markdown(mermaid_markdown))

# Alternatively, you can also save the mermaid syntax to a file for later use
with open("graph_diagram.mmd", "w") as f:
    f.write(mermaid_syntax)
print("Graph diagram syntax saved to 'graph_diagram.mmd'")


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	outline_agent(outline_agent)
	slide_agent(slide_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	supervisor -.-> outline_agent;
	supervisor -.-> slide_agent;
	supervisor -.-> __end__;
	outline_agent -.-> supervisor;
	slide_agent -.-> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```


Graph diagram syntax saved to 'graph_diagram.mmd'
